In [5]:
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
from torch.utils.data import Dataset
import torch
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from evaluate import load
import numpy as np
import matplotlib.pyplot as plt
from Bio.SeqIO.FastaIO import SimpleFastaParser 

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
BASE_TRAINER_KWARGS = {
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "logging_strategy": "epoch",
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "report_to": "none",
    "label_names": ["labels"]
}

In [4]:
model = AutoModelForTokenClassification.from_pretrained(
    'Synthyra/ESMplusplus_small',
    trust_remote_code=True,
    num_labels=2
).to(device)
tokenizer = model.tokenizer

Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
dataset = load_dataset("csv", data_files="data/deepcoil_train.csv")

Generating train split: 10438 examples [00:00, 99419.44 examples/s]


In [8]:
dataset['id']

KeyError: 'id'

In [46]:
raw_predictions = predictions_output.predictions[0]

# Set a threshold for your predictions
threshold = 0.5

# Iterate through the original labels and the raw predictions
for i in range(len(test_labels)):
    original_labels = test_labels[i]

    # Get the raw logits for the current sequence
    sequence_logits = raw_predictions[i]

    # Apply softmax to get probabilities for each class
    # The result will be a tensor of shape (sequence_length, 2)
    probabilities = softmax(torch.from_numpy(sequence_logits), dim=1)

    # Get the probabilities for class 1 (coiled-coil)
    # The second column (index 1) of the probabilities tensor corresponds to class 1
    class_1_probs = probabilities[:, 1]

    # Apply the threshold to generate predictions.
    # This creates a boolean tensor, which we convert to integers (0 or 1).
    thresholded_predictions = (class_1_probs >= threshold).int()

    # Convert the original and predicted labels to strings for printing
    original_labels_str = "".join(map(str, original_labels))
    predicted_labels_str = "".join(map(str, thresholded_predictions.tolist()[:len(original_labels)]))

    # Print the comparison
    print(f"Sequence {i+1}:")
    print(f"  Labels: {original_labels_str}")
    print(f"  Preds:  {predicted_labels_str}")
    print("-" * 20)

Sequence 1:
  Labels: 0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
  Preds:  0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
--------------------
Sequence 2:
  Labels: 0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000011111111111111111100000
  Preds:  0000000000000000000000000000000000000000000000000000000000000000000000000000011111111111111111111111111111111111100000
--------------------
Sequence 3:
  Labels: 000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

In [36]:
for i, original_labels in enumerate(test_labels):
    # Get the raw predictions (logits) for the current sequence
    sequence_logits = logits[i]

    # Find the class with the highest logit for this specific sequence
    predicted_labels = np.argmax(sequence_logits, axis=1)

    # Convert the original labels to a string
    original_labels_str = "".join(map(str, original_labels))

    # The prediction output might include predictions for special tokens.
    # We slice the predicted labels to match the length of the original labels.
    # This assumes the labels correspond to the actual sequence tokens.
    predicted_labels_str = "".join(map(str, predicted_labels[:len(original_labels)]))

    # Print the comparison
    print(f"Sequence {i+1}:")
    print(f"  Labels: {original_labels_str}")
    print(f"  Preds:  {predicted_labels_str}")
    print("-" * 20)

Sequence 1:
  Labels: 0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
  Preds:  [110 201][65 96][276 125][282 196][ 72 293][50  0][93 31][115  77][423 282][76 41][64 87][ 36 148][ 80 136][63  0][226 150][198 118][ 62 118][54  0][94 27][112  23][ 69 175][ 60 329][ 5 80][293 179][175   0][145 114][ 75 404][231 125][404  21][ 18 138][122 183][276  78][105 253][167  44][  9 217][46 65][290 177][52 18][156  71][192   4][194  25][127  61][ 66 130][111  28][236 343][181 121][ 64 147][ 69 146][162 304][ 81 143][100 235][212   0][68 95][ 98 316][88 36][263 286][152   0][ 67 127][ 0 16][ 32 119][38 79][129 228][20  0][ 0 47][10 53][66 82][82  0][ 72 119][60  2][180  79][179 432][15 77][238   0][306 314][172 347][64 27][80 95][258 123][412 324][255 113][94 48][ 88 242][194 278][172 107][111   0][1

IndexError: tuple index out of range